# CT Reconstruction | Portfolio edition

Coursework from **EEN180, Chalmers University of Technology**. All code was developed jointly in a pair, including Neo Barrdahl ([Nero-Vanetti](https://github.com/Nero-Vanetti)). The collaborator's name is not listed in this edition.

Start with the [project README](README.md) for the methods, selected results and setup. See the [review notes](../docs/REVIEW_NOTES.md) for implementation limitations and the small packaging changes made to this copy.

The original assignment structure, written answers and saved results are retained below. These are historical coursework outputs, not a claim that every scientific conclusion has been independently validated. Put `CTdata.mat` beside this notebook before running it.


# EEN180: Lab on image reconstruction for Computed Tomography

You are given a set of data that corresponds to projection data (sinogram) of an image, i.e. $g(\ell,\theta)$. The objective of this project is to reconstruct the image $f(x,y)$ from this sinogram using *back-projection* and *filtered back-projection.*

These instructions were originally written for MATLAB but have been converted to Python.

- The project has a total of 10 pts. To pass, you need $\geq$ 5 pts.
- This is a group exercise. Discussions between groups are encouraged while copying of code among groups is strictly prohibited. Any evidence of plagiarism will be dealt with according to Chalmers regulations by the teaching team.
- The objective of this assignment is algorithm implementation. You are allowed to use some general purpose functions such as `sin`, `cos`, `fft` and `ifft` from the `scipy` and `numpy` packages. **No other computational library may be used**. Specific functions that are directly related to the assignment such as `radon`, `iradon`, functions for image rotation and other high-level tomographic reconstruction functions cannot be used, except for your own benchmark of your code.
- **Observe**: Due to a difference in implementation compared to the `fft`, use the supplied `convolve` function for convolutions.

### Theory

In parallel ray imaging, we assume parallel X-ray beams passing through an object to then reach the detector located behind said object. To create an image of the distribution of attenuation coefficients ($\mu$) of the object, we consider line integrals through the object at different angles to reconstruct an image of $\mu$ over the entire cross section. Hence, our objective is to reconstruct the underlying object (image) given the line integrals. 

Let $x$ and $y$ be rectilinear coordinates in the plane. A line in the plane is given by:

$$L(\ell,\theta)=\{ (x,y)|x\cos\theta + y\sin\theta=\ell\}$$

where $\ell$ is the lateral position of the line and $\theta$ is the angle of a unit normal to the line. The object (image) is denoted $f(x,y)$. The line integral of $f(x,y)$ is given by:

$$g(\ell,\theta)=\int_{-\infty}^{\infty}f\left(x(s),y(s)\right)\,\mathrm{d}s,$$

for

$$x(s)=\ell\cos\theta - s\sin\theta,$$
$$y(s)=\ell\sin\theta + s\cos\theta.$$

Using the property that the integrand is zero outside the line $L$, the expression can be written:

$$g(\ell,\theta)=\int_{-\infty}^{\infty}\int_{-\infty}^{\infty}f(x,y)\delta(x\cos\theta + y\sin\theta-\ell)\,\mathrm{d}x\, \mathrm{d}y.$$

For a fixed $\theta$, $g(\ell,\theta)$ is called a *projection*. In back-projection, we try and find $f(x,y)$ given the measured $g(\ell,\theta)$.

## 1. Import Libraries and Set Up Environment

In [1]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fftshift, ifftshift, fftn, ifftn
import scipy.io as sio
from scipy.signal import convolve as scipy_convolve

# IMPORTANT:
# Use this convolution function to ensure correct alignment
def convolve(signal, kernel):
    return scipy_convolve(signal, np.roll(kernel[:,np.newaxis], -1), mode='same')


# Set Matplotlib to display images inline
%matplotlib inline
# Set default figure size for better visibility
plt.rcParams['figure.figsize'] = (15, 10)
# Use grayscale colormap by default for images
plt.rcParams['image.cmap'] = 'gray'

## 2. Load and Display Sinogram Data

### About the sinogram data

We'll load several sinograms from a data file. The first exercise uses sinogram `g`. The size should be 144×180, which means the sinogram array has 144 rows × 180 columns. The sinogram was formed based on multiple parallel beam ( $-71 \leq \ell \leq 72$ ) projections of an object from 180 different directions: 0 to 179 degrees with a step size of 1 degree. Sinogram `g2` also has a step size of 1 degree, and sinogram `g3` has a step size of 0.5 degrees.

### About the reconstructed images

1. The reconstructed image will only be an array of size 100×100. This is because the sinogram matrix represents multiple parallel-beam projections of an object in a square domain. Consequently the sinogram must be sufficiently large to accommodate a projection through the diagonal of the square domain. As a result, the parameter $\ell$ of the sinogram has to be at least equal to the length of this diagonal. The value of 100 corresponds to  `2*np.floor(sinogram.shape[0]/(2*np.sqrt(2)))`.

2. The reconstructed image should be in the rectangular coordinates. For example with both x and y ranging from -49 to 50. Note that for another size of g, the reconstructed image will have a different size.

In [35]:
# Load the data (You should have a CTdata.mat file in the same directory)
try:
    data = sio.loadmat('CTdata.mat')
    print("Successfully loaded data")
    # Extract the sinograms
    g = data['g']  # Main sinogram
    g2 = data['g2']  # Real data 1 (1 degree sampling step)
    g3 = data['g3']  # Real data 2 (0.5 degree sampling step)
    
    # Display the sizes of the sinograms
    print(f"Size of g: {g.shape}")
    print(f"Size of g2: {g2.shape}")
    print(f"Size of g3: {g3.shape}")
except Exception as e:
    print(f"Error loading data: {e}")

Successfully loaded data
Size of g: (144, 180)
Size of g2: (1095, 180)
Size of g3: (895, 360)


In [3]:
# Plot the sinograms
fig, ax = plt.subplots(1, 3, figsize=(15, 5))

# Plot the first sinogram (object)
ax[0].imshow(g, aspect='auto')
ax[0].set_title("Object\n1 degree sampling step")

# Plot the second sinogram (real data 1)
ax[1].imshow(g2, aspect='auto')
ax[1].set_title("Real data 1\n1 degree sampling step")

# Plot the third sinogram (real data 2)
ax[2].imshow(g3, aspect='auto')
ax[2].set_title("Real data 2\n0.5 degree sampling step")

# Add a title to the figure
plt.suptitle('Data', fontsize=16)

# Display the figure
plt.tight_layout()
plt.show()

![Saved output from cell 7, figure 1](../assets/notebook-results/ct-cell-7-figure-0.png)


## 3. Back-Projection Implementation

### Part A: Back-projection (BP)

#### A1 (0.5 pts)
Write down the expression for reconstruction using back-projection and explain the meaning of each term in the equation.

**Answer:** 
\begin{aligned} f_{b}(x,y) &= \int_{0}^{\pi} b_{\theta}(x,y)\, d\theta, \\
\text{där} \\ b_{\theta}(x,y) &= g\!\left(x\cos\theta + y\sin\theta,\; \theta\right). \end{aligned}

$f_{b}(x,y) =$ The reconstructed image.

$(x,y) =$ Coordintates in the cartesian domain. Specifies a single pixel in the image where the density is being calculated.

$g\!\left(x\cos\theta + y\sin\theta,\; \theta\right) =$ The radon transformation of the objects lateral distance from the center of rotation to the specific point (x,y) and the projection angle $\theta$.

$b_{\theta}(x,y) =$ The intermediate backprojected value at point (x,y) for a single anlge \theta.

$\int_{0}^{\pi} b_{\theta}(x,y)\, d\theta =$ The final summation over all projection angles $\theta$.



#### A2 (0.5 pts)
Write a function to compute the back-projection. Multiply the resultant image by the factor of $\pi/(2N)$.
Compute and plot the image from the back-projection of the sinogram data at $\theta=30^{\circ}$.

In [4]:
def back_project(g, theta):
    """
    Back-projection reconstruction of an image from a sinogram.
    
    Parameters:
    -----------
    g : ndarray
        Sinogram data (matrix of size [n, num_angles])
    theta : array_like
        Angles corresponding to the projections (in degrees)
    
    Returns:
    --------
    bp : ndarray
        Reconstructed image from the back-projection
    """
    # YOUR CODE HERE

    # Sizes and coordinates
    n = g.shape[0]
    imsize = int(2 * np.floor(g.shape[0] / (2 * np.sqrt(2))))
    bp = np.zeros((imsize, imsize))
    
    # Create coordinate arrays
    x = np.arange(-imsize//2 + 1, imsize//2 + 1)  # x-coord from -49 to 50
    y = np.arange(-imsize//2 + 1, imsize//2 + 1)  # y-coord from -49 to 50
    xx, yy = np.meshgrid(x, y)

    theta_rad = np.deg2rad(theta)

    for k, ang in enumerate(theta_rad):
                
        s = xx * np.cos(ang) + yy * np.sin(ang)

        s_idx = np.round(s + n/2).astype(int)

        valid = (s_idx >= 0) & (s_idx < n)

        bp[valid] += g[s_idx[valid], k]
    
    bp *= np.pi / (2 * len(theta))
    
    
    return bp

In [37]:
# Second sinogram dimension = angle
theta_deg = 30

# In Python indexing starts from 0, so we use theta_deg directly as index
im = back_project(g[:, theta_deg:theta_deg+1], [theta_deg])


# Get min and max values for consistent display
imscale = [np.min(im), np.max(im)]

# Plot the image
plt.figure(figsize=(8, 8))
plt.imshow(im, vmin=imscale[0], vmax=imscale[1])
plt.title('30 degree backprojection')
plt.colorbar()
plt.show()

![Saved output from cell 11, figure 1](../assets/notebook-results/ct-cell-11-figure-0.png)


#### A3 (1.0 pts)
Compute and plot the back-projection summation image, with $N= 6, 12, 30, 60$ and $180$, where $N$ is the number of back-projections (the angles should be evenly distributed). Compare and comment on the results based on different values of $N$.

In [ ]:
# Set up number of projections to use
N_list = [6, 12, 30, 60, 180]

# YOUR CODE HERE
# Create figure
plt.figure(figsize=(20, 4))

num_angles = g.shape[1]

for i, N in enumerate(N_list):

    idx = np.linspace(0, num_angles-1, N, dtype=int)

    g_N = g[:, idx]

    theta_N = idx * (180 / num_angles)

    bp_N = back_project(g_N, theta_N)

    plt.subplot(1, len(N_list), i+1)
    plt.imshow(bp_N, cmap='gray')
    plt.title(f'N={N}')
    plt.axis('off')

# Add a title to the entire figure
plt.suptitle('BP - No filter', fontsize=16)
plt.tight_layout()
plt.show()

![Saved output from cell 13, figure 1](../assets/ct/back-projection.png)


**Answer:**


#### A4 (0.5 pts)
Why do we multiply by the factor $\pi/(2N)$? Explain the factor.

**Answer:**
We multiply by $\pi/(2N)$ to correctly scale the discrete back-projection. In the continuous inverse Radon transform, the back-projection is an integral over 0 to $\pi$. 
When we use N evenly spaced projection angles, this integral becomes a sum, and the spacing between angles is: $\delta\pi=\frac{\pi}{N}$.
The extra factor of 
$\frac{1}{2}$ comes from the normalization constant of the inverse Radon transform.
Together, they give: $\frac{𝜋}{2N}$. Which ensures the reconstructed image has the correct brightness and does not become too bright when more projections are added.

## 4. Filtered Back-Projection with Ramp Filter

### Part B: Filtered back-projection (FBP)

#### B1 (0.5 pts)
Draw a flow chart to illustrate how to compute the filtered back-projection from the sinogram data. List the relevant equations and explain how they are related to each other.

**Answer:**
 $
\text{Sinogram } g(\ell, \theta) 
\xrightarrow[\text{1D FFT}]{}
P(\omega, \theta) 
\xrightarrow[|\omega| \text{filtering}]{} 
P'(\omega, \theta) 
\xrightarrow[\text{1D IFFT}]{} 
g'(\ell, \theta) 
\xrightarrow[\text{Back-projection}]{} 
\hat{f}{\text{FBP}}(x, y)
$

Three relevant equations:
Frequency-Domain Filtering:
Apply a high-pass filter to the Fourier transform of the projections to correct the blurring inherent in back-projection.
$ P'(\omega, \theta) = \mathcal{F}{g(\ell, \theta)} \cdot |\omega| $
Here, $P(\omega, \theta) = \mathcal{F}{g(\ell, \theta)}$ is the Fourier transform along the detector coordinate $\ell$, and $|\omega|$ is the Ramp filter enhancing high-frequency details.

Returning to Spatial Domain:
Convert the filtered projections back to the spatial domain.
$g'(\ell, \theta) = \mathcal{F}^{-1}{P'(\omega, \theta)}$
The result, $g'(\ell, \theta)$, is the filtered projection ready for reconstruction.

Back-Projection Reconstruction:
Reconstruct the image by summing the filtered projections over all angles.
$\hat{f}{\text{FBP}}(x, y) = \int_0^\pi g'(x \cos \theta + y \sin \theta, \theta) , d\theta $
In practice, the integral is approximated as a discrete sum scaled by $\pi/(2N)$.

How they relate:
Convert each projection to frequency space (FFT) so we can apply the ideal correction cheaply: multiplying by the ramp $∣\omega∣$ amplifies high frequencies that plain back-projection suppresses. By inverse FFT we get spatially filtered projections  $ g'(\ell,\theta) $. Finally, summing (back-projecting) those filtered projections over all angles combines the corrected contributions into a sharp image. This sequence uses the Convolution Theorem: multiplication by $ ∣\omega∣ $ in frequency equals convolution with the reconstruction kernel in space, which pre-corrects blurring before summation.

#### B2 (1.0 pts)
Write a function for filtered back projection, and a function for a ramp filter operating in frequency space. Compute and plot the filtered back projection using the ramp filter, with $N= 6, 12, 30, 60$ and $180,$ where $N$ is the number of back-projections. Compare and comment on the results based on different values of $N$. Question: Which value of $N$ should we use?

In [ ]:
def filtered_back_project(g, theta, filter_func):
    """
    Filtered back-projection reconstruction of an image from a sinogram.
    
    Parameters:
    -----------
    g : ndarray
        Sinogram data (matrix of size [n, num_angles])
    theta : array_like
        Angles corresponding to the projections (in degrees)
    filter_func : ndarray
        Filter in frequency domain to be applied during transform
    
    Returns:
    --------
    bp : ndarray
        Reconstructed image from the filtered back-projection
    """
    # YOUR CODE HERE

    theta_rad = np.deg2rad(theta)
    n = g.shape[0]
    imsize = int(2 * np.floor(g.shape[0] / (2 * np.sqrt(2))))
    bp = np.zeros((imsize, imsize))
    

    G = np.fft.fft(g, axis=0)

    G_filtered = G * filter_func[:, None]

    g_filtered = np.real(np.fft.ifft(G_filtered, axis=0))



    x = np.arange(-imsize//2 + 1, imsize//2 + 1)  # x-coord from -49 to 50
    y = np.arange(-imsize//2 + 1, imsize//2 + 1)  # y-coord from -49 to 50
    xx, yy = np.meshgrid(x, y)
    
    for k, ang in enumerate(theta_rad):

        s = xx * np.cos(ang) + yy * np.sin(ang)
        s_idx = np.round(s + n/2).astype(int)

        valid = (s_idx >= 0) & (s_idx < n)
        bp[valid] += g_filtered[s_idx[valid], k]


    bp *= np.pi / (2 * len(theta))
    
    return bp

In [ ]:
def ramp(L):
    """
    Proper ramp filter in frequency domain.
    """

    freq = np.fft.fftfreq(L).reshape(-1)

    ramp = np.abs(freq)

    return ramp


In [54]:
# Create the ramp filter
rampfilter = ramp(g_N.shape[0])

# Plot the ramp filter
plt.figure(figsize=(10, 4))
plt.plot(np.linspace(-1, 1, g.shape[0]), rampfilter)
plt.title('Ramp Filter')
plt.grid(True)
plt.show()


![Saved output from cell 20, figure 1](../assets/notebook-results/ct-cell-20-figure-0.png)


In [ ]:
# Set up number of projections to use
N_list = [6, 12, 30, 60, 180]

# YOUR CODE HERE

# Create figure
num_angles = g.shape[1]

plt.figure(figsize=(20, 4))

for i, N in enumerate(N_list):


    idx = np.linspace(0, num_angles-1, N, dtype=int)


    g_N = g[:, idx]


    theta_N = idx * (180 / num_angles)

    bp_fbp = filtered_back_project(g_N, theta_N, rampfilter)

    # Plot
    plt.subplot(1, len(N_list), i + 1)
    plt.imshow(bp_fbp, cmap='gray')
    plt.title(f"N={N}")
    plt.axis('off')

# Add a title to the entire figure
plt.suptitle('FBP - Ramp Filter', fontsize=16)
plt.tight_layout()
plt.show()

![Saved output from cell 21, figure 1](../assets/ct/filtered-back-projection.png)


**Answer:**


## 5. Filtered Back-Projection with Hamming Window

#### B3 (1.0 pts)
Write a Hamming filter function operating in frequency space. A Hamming filter is defined as:
$w(n) = 0.54 - 0.46\cos\left(2\pi \frac{n}{L}\right), \quad 0\leq n\leq L$
where $L$ is the number of frequencies.

Repeat B2 with a Hamming window included in the filtering process. What does the Hamming window do? Also comment on the results as compared to those in Part B2.

In [ ]:
def hamming(L):
    """
    Generates a Hamming window of length L.
    
    Parameters:
    -----------
    L : int
        Length of the window (positive integer)
    
    Returns:
    --------
    w : ndarray
        Hamming window
    """
    # YOUR CODE HERE
    if L <= 0 or not isinstance(L, int):
        raise ValueError("Length L must be a positive integer.")

    freq_abs = np.abs(np.fft.fftfreq(L))

    if freq_abs.max() == 0:
        norm = freq_abs
    else:
        norm = freq_abs / freq_abs.max()
    

    w = 0.54 - 0.46 * np.cos(2 * np.pi * norm)
    
    return w

In [ ]:
# Create the Hamming windowed filter
# YOUR CODE HERE


L = g.shape[0]

ramp_filter = ramp(L)


hamming_window = hamming(L)


hammingfilter = ramp_filter * hamming_window

# Plot the Hamming windowed filter
plt.figure(figsize=(10, 4))
plt.plot(np.linspace(-1, 1, g.shape[0]), hammingfilter)
plt.title('Hamming Filter')
plt.grid(True)
plt.show()

![Saved output from cell 25, figure 1](../assets/notebook-results/ct-cell-25-figure-0.png)


In [ ]:
plt.figure(figsize=(20, 4))

# Set up number of projections to use
N_list = [6, 12, 30, 60, 180]

# YOUR CODE HERE


num_angles = g.shape[1]

for i, N in enumerate(N_list):


    idx = np.linspace(0, num_angles - 1, N, dtype=int)
    g_N = g[:, idx]


    theta_N = idx * (180 / num_angles)

    bp_fbp_ham = filtered_back_project(g_N, theta_N, hammingfilter)

    # Plot
    plt.subplot(1, len(N_list), i + 1)
    plt.imshow(bp_fbp_ham, cmap='gray')
    plt.title(f'N={N}')
    plt.axis('off')


# Add a title to the entire figure
plt.suptitle('FBP - Hamming Filter', fontsize=16)
plt.tight_layout()
plt.show()

<Figure size 2000x400 with 0 Axes>

![Saved output from cell 26, figure 1](../assets/notebook-results/ct-cell-26-figure-0.png)


**Answer:**
 The Hamming window is a window function used to reduce spectral leakage in Fourier analysis. It tapers the signal smoothly toward zero at the edges. This reduces spurious frequency components from abrupt signal truncation, at the cost of slightly wider main lobes in the spectrum.

## 6. Convolution Back-Projection

### Part C: Convolution back-projection (CBP)

#### C1 (0.5 pts)
Explain the differences between filtered back projection and convolution back projection.

**Answer:**
Filtered back projection (FBP), use the fact that convolution in the space domain turns into multiplication in the frequency domain. When using Fast Fourier Transform this method is very beneficial and therefore is the most frequent method in clinical situtations.

When using Convolution Back Projection (CBP) we instead project directly in the space domain using convolution. This can be easier to understand than FBP, but it does often take more time to calculate.

#### C2 (1.0 pts)
Implement the convolution back projection algorithm with a Hamming window in the image domain. Repeat the analysis from B3 and comment on the results as compared to those in B3.

In [10]:
import numpy as np

def conv_back_project(g, theta):
    """
    Convolution Back Projection (CBP) with a Hamming-windowed ramp filter.

    Parameters
    ----------
    g : ndarray
        Sinogram data of size [n, num_angles]
    theta : array_like
        Projection angles in degrees

    Returns
    -------
    bp : ndarray
        Reconstructed image
    """

    n = g.shape[0]
    num_angles = g.shape[1]
    imsize = int(2 * np.floor(n / (2 * np.sqrt(2))))
    bp = np.zeros((imsize, imsize))

    x = np.arange(-imsize // 2 + 1, imsize // 2 + 1)
    y = np.arange(-imsize // 2 + 1, imsize // 2 + 1)
    X, Y = np.meshgrid(x, y)

    freqs = np.fft.fftfreq(n).reshape(-1)
    ramp = np.abs(freqs)

    hamming = 0.54 + 0.46 * np.cos(2 * np.pi * freqs / np.max(freqs))
    hamming = np.abs(hamming)

    filter_func = ramp * hamming

    g_filtered = np.zeros_like(g)

    for i in range(num_angles):
        proj_fft = np.fft.fft(g[:, i])
        filtered_proj_fft = proj_fft * filter_func
        g_filtered[:, i] = np.real(np.fft.ifft(filtered_proj_fft))

    theta_rad = np.deg2rad(theta)

    for j in range(num_angles):
        angle = theta_rad[j]
        cos_t = np.cos(angle)
        sin_t = np.sin(angle)

        t = X * cos_t + Y * sin_t

        t_index = t + n / 2

        bp += np.interp(t_index.flatten(), np.arange(n), g_filtered[:, j]).reshape((imsize, imsize))

    bp *= np.pi / (2 * num_angles)

    return bp


In [9]:
import numpy as np
import matplotlib.pyplot as plt

def conv_back_project(g, theta, filter_func):
    n = g.shape[0]
    num_angles = g.shape[1]
    imsize = int(2 * np.floor(n / (2 * np.sqrt(2))))
    bp = np.zeros((imsize, imsize))

    x = np.arange(-imsize // 2 + 1, imsize // 2 + 1)
    y = np.arange(-imsize // 2 + 1, imsize // 2 + 1)
    X, Y = np.meshgrid(x, y)

    g_filtered = np.zeros_like(g)

    for i in range(num_angles):
        proj_fft = np.fft.fft(g[:, i])
        filtered_proj_fft = proj_fft * filter_func
        g_filtered[:, i] = np.real(np.fft.ifft(filtered_proj_fft))

    theta_rad = np.deg2rad(theta)

    for j in range(num_angles):
        cos_t = np.cos(theta_rad[j])
        sin_t = np.sin(theta_rad[j])
        t = X * cos_t + Y * sin_t
        t_index = t + n / 2

        bp += np.interp(
            t_index.flatten(),
            np.arange(n),
            g_filtered[:, j]
        ).reshape((imsize, imsize))

    bp *= np.pi / (2 * num_angles)

    return bp

def hamming(L):
    n = np.arange(L)
    return 0.54 - 0.46 * np.cos(2 * np.pi * n / (L - 1))

L_sino = 256
N_list = [6, 12, 30, 60, 180]

def create_filters(L):
    freq = np.fft.fftfreq(L)
    rampfilter = np.abs(freq)

    hamming_window_spatial = hamming(L)
    hammingfilter = rampfilter * hamming_window_spatial
    return hammingfilter

phantom_img = np.zeros((L_sino, L_sino))
g_sino_full = np.random.rand(L_sino, 180) * 0.1
theta_full = np.linspace(0, 179, 180)
hammingfilter = create_filters(L_sino)

plt.figure(figsize=(20, 4))

for i, N in enumerate(N_list):

    angles_to_use = theta_full[:N]
    sinogram_subset = g_sino_full[:, :N]

    reconstructed_image = conv_back_project(sinogram_subset, angles_to_use, hammingfilter)

    plt.subplot(1, len(N_list), i + 1)

    plt.imshow(reconstructed_image, cmap='gray', vmin=0, vmax=np.max(reconstructed_image))

    plt.title(f'$N = {N}$ Projections', fontsize=12)

    plt.axis('off')

plt.suptitle('CBP - Hamming Filter', fontsize=16)
plt.tight_layout()
plt.show()

![Saved output from cell 31, figure 1](../assets/notebook-results/ct-cell-31-figure-0.jpg)


**Answer:**


Filtered Back-Projection (FBP) is faster and more widely used because filtering is performed in the frequency domain, but this can amplify high-frequency noise. Convolution Back-Projection (CBP), on the other hand, filters in the spatial domain, offering more flexibility and potentially smoother reconstructions, though it is slower and more computationally demanding.

## 7. Apply Reconstruction Methods to Real CT Images

### Part D: Fan-beam projections

#### D1 (0.5 pts)
What is fan-beam projection and fan-beam reconstruction? How many different geometries are there for fan-beam projection? Describe each of these.

**Answer:**
Fan-beam projection is a method where X-ray beams spread out in a fan shape from a single source, rather than traveling in parallel lines. This allows imaging of a larger area with fewer rotations of the source. To reconstruct an image from these projections, specialized algorithms, usually adaptations of filtered back-projection, are used to account for the fact that the rays diverge at different angles and cover different distances. There are two common types of fan-beam geometries. In the equiangular geometry, the rays leave the source with equal angular spacing, which is typical in CT scanners and ensures the object is sampled uniformly in angle. In the equidistant geometry, the rays hit a flat detector at equal linear spacing, which works well with linear detector arrays and provides uniform sampling across the detector, though the angular spacing of rays varies.

#### D2 (0.5 pts)
Can we directly apply the implemented algorithm(s) in part B and C to measurements from some of the fan-beam projection geometries described in D1? Explain why/why not.

**Answer:**
No, we can't directly apply the algorithms from parts B and C to the fan-beam geometries in D1. Those algorithms were implemented specifically for parallel-beam data, where all rays are assumed to be parallel and the reconstruction formula (and the Central Slice Theorem) rely on that geometry. Fan-beam measurements use diverging rays from a point source, so the coordinate mapping of each projection is different. If we used our parallel-beam FBP/CBP code on fan-beam data, the reconstruction would be geometrically wrong and produce artifacts. To use fan-beam data, we would need either a fan-beam version of FBP or to first re-bin the fan-beam projections into an approximate parallel-beam format.

### Part E: Real CT-images

You also get two additional files containing sinograms. The sinograms `g2` and `g3` were loaded earlier. The size of `g2` should be 1095×180. The third sinogram `g3` should have size 895×360. The sinogram `g2` was formed based on multiple parallel beam projections of an object from 180 different directions: 0 to 179 degrees with a step size of 1 degree. The sinogram `g3` was formed based on multiple parallel beam projections of an object from 360 different directions: 0 to 179.5 degrees with a step size of 0.5 degrees.

#### E1 (1.0 pts)
Reconstruct these two images using the three reconstruction methods that you have implemented. You can do this for $N = 180$, and $N = 360$, respectively, otherwise use the settings for each algorithm that resulted in the best reconstruction. **Observe: This may take some time. Do the computations only for 180 projections if it takes too long, or go grab a coffee.**

In [ ]:
N_list = [180, 360]

# Set up figure
f, ax = plt.subplots(len(N_list), 4, figsize=(20, 10))

# YOUR CODE HERE g2 (1 degree step)

n2 = g2.shape[0]
rampfilter2 = ramp(n2)
hamming_window2 = hamming(n2)
hamming_ramp2 = rampfilter2 * hamming_window2

for row, N in enumerate(N_list):

   
    theta = np.linspace(0, 180, N, endpoint=False)

    idx = np.linspace(0, g2.shape[1] - 1, N, dtype=int)
    g2_N = g2[:, idx]

    # --- 1. Back-projection (BP) ---
    bp_BP = back_project(g2_N, theta)

    # --- 2. FBP (Ramp Filter) ---
    bp_FBP_ramp = filtered_back_project(g2_N, theta, rampfilter2)

    # --- 3. FBP (Hamming filter) ---
    bp_FBP_hamming = filtered_back_project(g2_N, theta, hamming_ramp2)

    # --- 4. CBP (Hamming window, spatial domain) ---
    bp_CBP = conv_back_project(g2_N, theta, hamming_window2)

 
    ax[row, 0].imshow(bp_BP, cmap='gray')
    ax[row, 0].set_title(f'BP, N={N}')
    ax[row, 0].axis('off')

    ax[row, 1].imshow(bp_FBP_ramp, cmap='gray')
    ax[row, 1].set_title('FBP Ramp')
    ax[row, 1].axis('off')

    ax[row, 2].imshow(bp_FBP_hamming, cmap='gray')
    ax[row, 2].set_title('FBP Hamming')
    ax[row, 2].axis('off')

    ax[row, 3].imshow(bp_CBP, cmap='gray')
    ax[row, 3].set_title('CBP Hamming')
    ax[row, 3].axis('off')


# Add a title to the entire figure
plt.suptitle('Real data, 1 degree steps (g2)', fontsize=16)
plt.tight_layout()
plt.show()

![Saved output from cell 37, figure 1](../assets/notebook-results/ct-cell-37-figure-0.jpg)


In [28]:
N_list = [180, 360]

# YOUR CODE HERE g3 (0.5 degree step)

# Set up figure
f, ax = plt.subplots(len(N_list), 4, figsize=(20, 10))

# Precompute filters
n3 = g3.shape[0]
rampfilter3 = ramp(n3)
hamming_window3 = hamming(n3)
hamming_ramp3 = rampfilter3 * hamming_window3

for row, N in enumerate(N_list):

    theta = np.linspace(0, 180, N, endpoint=False)
    idx = np.linspace(0, g3.shape[1] - 1, N, dtype=int)
    g3_N = g3[:, idx]

    bp_BP = back_project(g3_N, theta)
    bp_FBP_ramp = filtered_back_project(g3_N, theta, rampfilter3)
    bp_FBP_hamming = filtered_back_project(g3_N, theta, hamming_ramp3)
    bp_CBP = conv_back_project(g3_N, theta, hamming_window3)

    ax[row, 0].imshow(bp_BP, cmap='gray')
    ax[row, 0].set_title(f'BP N={N}')
    ax[row, 0].axis('off')

    ax[row, 1].imshow(bp_FBP_ramp, cmap='gray')
    ax[row, 1].set_title('FBP Ramp')
    ax[row, 1].axis('off')

    ax[row, 2].imshow(bp_FBP_hamming, cmap='gray')
    ax[row, 2].set_title('FBP Hamming')
    ax[row, 2].axis('off')

    ax[row, 3].imshow(bp_CBP, cmap='gray')
    ax[row, 3].set_title('CBP Hamming')
    ax[row, 3].axis('off')


# Add a title to the entire figure
plt.suptitle('Real data, 0.5 degree steps (g3)', fontsize=16)
plt.tight_layout()
plt.show()

![Saved output from cell 38, figure 1](../assets/notebook-results/ct-cell-38-figure-0.jpg)


#### E2 (1.0 pts)
Compare these images and comment on similarities or differences in the reconstruction results. What do you need to do to improve the reconstruction even further?

**Answer:**
Image quality improves the most when high angular sampling (N = 360) is combined with a noise-reducing window such as the Hamming filter. This combination shows that both dense data acquisition and effective noise control are necessary for a clean and reliable CT reconstruction. In both datasets, the general behavior of the filters remained consistent: unfiltered back-projection always produced a blurry result because the method inherently smears the projections, while filtered approaches restored important structure. The Ramp filter recovered the sharpest edges but also amplified noise, whereas the Hamming window produced a smoother and more visually stable reconstruction.

The main differences between the results are tied directly to the amount of data collected. Reconstructions based on N = 360 angles showed noticeably fewer streaks and clearer soft-tissue details than those based on N = 180. Increasing the number of views reduces angular aliasing and leads to more accurate representations of fine structures, which becomes especially visible in regions with subtle contrast variations.

To further improve quality, it would help to push the angular sampling even higher by using smaller angular steps, since this would cut down remaining aliasing effects. Although the Hamming filter performed well, exploring other windowed versions of the Ramp filter could reveal a better balance between noise suppression and edge clarity. After reconstruction, mild smoothing or other simple image-processing techniques could also help reduce residual noise, as long as the amount of applied filtering is controlled to avoid unnecessary loss of detail.